In [ ]:
from pathlib import Path

import napari
import numpy as np
from natsort import natsorted
from tifffile import imread
from tqdm import tqdm

In [ ]:
base_path = Path(
    r"/mnt/home/hoatman/ceph/lightsheet_20250131/raw_image/downscaled/recon"
)

arr_out = []

for fp in tqdm(natsorted(list(base_path.glob("*.tif")))):
    arr_out.append(imread(fp))

In [ ]:
import h5py

with h5py.File("/mnt/home/hoatman/ceph/lightsheet_20250131/recon.h5", "w") as f:
    for i, arr in tqdm(enumerate(arr_out)):
        f.create_dataset(f"frame_{i: 03d}", data=arr)

In [ ]:
test_load = []

with h5py.File("/mnt/home/hoatman/ceph/lightsheet_20250131/recon.h5", "r") as f:
    for dset in tqdm(f.keys()):
        loaded_data = f[dset][()]
        test_load.append(loaded_data)

In [ ]:
print(arr_out[0].shape)

In [ ]:
arr_out = np.array(arr_out)

In [ ]:
viewer = napari.Viewer()


viewer.add_image(arr_out, scale=(0.5825, 0.5825, 0.5825))

napari.run()

In [ ]:
embryo_path = Path(r"/mnt/home/hoatman/ceph/lightsheet_20250131")
napari_out_path = embryo_path / "napari"
napari_out_path.mkdir(exist_ok=True)

for frame in range(arr_out.shape[0]):
    viewer.dims.set_point(0, frame)
    viewer.screenshot(
        napari_out_path / f"frame_{frame:04d}.tif",
        canvas_only=True,
        scale=1,
        flash=False,
    )